# Fine-tuning de RoBERTa biomédico-clínico para el NER de AI-Triage (paso 3.1)

**Modelo base:** [`PlanTL-GOB-ES/roberta-base-biomedical-clinical-es`](https://huggingface.co/PlanTL-GOB-ES/roberta-base-biomedical-clinical-es) (Apache-2.0); réplica en `BSC-LT/roberta-base-biomedical-clinical-es`.

**Etiquetas:** `SINTOMA`, `ENFERMEDAD` y `FARMACO`, en esquema BIO. El modelo solo detecta menciones. Si una mención está negada, es un antecedente o es una alergia lo decide el pipeline después (`negation.py` y reglas de contexto).

| Etapa | Datos | Para qué |
| :--- | :--- | :--- |
| 1 | SympTEMIST + DisTEMIST + DrugTEMIST (CC BY 4.0), fusionados por documento; PharmaCoNER opcional | Aprender las tres entidades en casos clínicos escritos |
| 2 | Split `dev` del corpus propio (150 transcripciones, lotes 1 y 2) | Adaptar al habla transcrita de urgencias |

**Reglas de evaluación (no negociables):**
- El split `test` del corpus propio **no se usa aquí**. Se evalúa una sola vez en el paso 4.1.
- La etapa 2 se mide con **validación cruzada de 5 particiones sobre `dev`**. El modelo final se entrena con todo `dev`.
- Semilla fija (`SEED`) e hiperparámetros guardados en `training_config.json`.

**Entorno:** Colab o Kaggle con GPU T4. Activa la GPU antes de ejecutar (*Entorno de ejecución → Cambiar tipo → T4*). Duración estimada: entre 30 y 45 minutos.

In [ ]:
# 1. Dependencias y GPU
!pip install -q "transformers>=4.44,<5" "datasets>=2.19" seqeval accelerate
import torch, transformers, datasets
print('torch', torch.__version__, '| transformers', transformers.__version__, '| datasets', datasets.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO HAY GPU: activa una T4 antes de seguir')
SEED = 42

## 2. Código y corpus propio

El repositorio es privado, así que hay dos opciones:
- **A. Paquete (recomendado).** En tu equipo, ejecuta `python ml/ner/make_bundle.py` y sube `ml/ner/output/ner_bundle.zip` cuando la celda lo pida. En Kaggle, súbelo como *Dataset* y ajusta `BUNDLE`.
- **B. Clonar.** Guarda un token de GitHub de solo lectura en los secretos de Colab con el nombre `GITHUB_TOKEN`.

In [ ]:
import os, sys, zipfile
from pathlib import Path

WORK = Path('/content' if Path('/content').exists() else '/kaggle/working')
CODE = WORK / 'ner_code'
CODE.mkdir(parents=True, exist_ok=True)
BUNDLE = Path('/kaggle/input/ner-bundle/ner_bundle.zip')  # Kaggle: ruta del dataset subido

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if token:
    !git clone -q --depth 1 https://{token}@github.com/kevinJ1910/ai-triage.git {WORK}/repo
    for name, rel in {'ner_data.py': 'ml/ner/ner_data.py', 'train_ner.py': 'ml/ner/train_ner.py',
                      'corpus_entidades.json': 'backend/triage/benchmark/corpus_entidades.json'}.items():
        (CODE / name).write_bytes((WORK / 'repo' / rel).read_bytes())
else:
    if not BUNDLE.exists():
        from google.colab import files
        uploaded = files.upload()  # selecciona ner_bundle.zip
        BUNDLE = WORK / next(iter(uploaded))
    zipfile.ZipFile(BUNDLE).extractall(CODE)

sys.path.insert(0, str(CODE))
import ner_data, train_ner
print(sorted(p.name for p in CODE.iterdir()))

## 3. Corpus públicos (Zenodo, CC BY 4.0)

Se verifica el MD5 publicado por Zenodo. SympTEMIST ocupa unos 450 MB porque incluye el *background set* y traducciones, que no se usan. DrugTEMIST viene dentro del ZIP de MultiCardioNER (pista 2).

**FARMACO sale de DrugTEMIST.** Sus guías anotan medicamentos, como `medications` del corpus propio. PharmaCoNER también anota sustancias químicas, proteínas y genes. Se puede sumar con `USE_PHARMACONER = True`, pero entonces solo quedan los documentos presentes en los cuatro corpus.

In [ ]:
import hashlib, re, urllib.request

DATASETS = {
    'distemist': ('https://zenodo.org/api/records/7614764/files/distemist_zenodo.zip/content', 'bc84461ad07b59a94c79ca9c648952c2'),
    'pharmaconer': ('https://zenodo.org/api/records/4270158/files/pharmaconer.zip/content', 'e511314c4468c2655ae3355ce590a2c0'),
    'symptemist': ('https://zenodo.org/api/records/10635215/files/symptemist-complete_240208.zip/content', 'd31f61b68cf111cb96790301e5adca33'),
    'multicardioner': ('https://zenodo.org/api/records/11368861/files/multicardioner_train+dev+test+bg+mappings_240528.zip/content', '56055c807e2cc329ff5eeaf7e4f713ba'),
}
USE_PHARMACONER = False
DATA = WORK / 'datos'
DATA.mkdir(exist_ok=True)

def md5(path):
    digest = hashlib.md5()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

for name, (url, checksum) in DATASETS.items():
    archive = DATA / f'{name}.zip'
    if not archive.exists() or md5(archive) != checksum:
        print('Descargando', name)
        urllib.request.urlretrieve(url, archive)
    assert md5(archive) == checksum, f'MD5 incorrecto en {name}'
    if not (DATA / name).exists():
        zipfile.ZipFile(archive).extractall(DATA / name)
    print(name, 'OK')

In [ ]:
# 4. Localizar los ficheros. La estructura interna cambia entre versiones: se buscan por patrón y se muestran.
from collections import Counter
find = ner_data.find_files

dis_tsvs = [p for p in find(DATA / 'distemist', '*subtrack1*mentions*.tsv')]
pharma_dirs = sorted({p.parent for p in find(DATA / 'pharmaconer', '*.ann') if 'subtrack1' in str(p)})
sym_tsvs = [p for p in find(DATA / 'symptemist', '*.tsv')
            if not ner_data._EXCLUDED_DIRS.search(str(p)) and 'ner' in str(p).lower()
            and 'linking' not in str(p).lower() and 'norm' not in p.name.lower()]
# DrugTEMIST: carpetas brat de la pista de medicamentos; se excluyen las de ingles e italiano y el background.
LANG_DIR = re.compile(r'(^|[/_-])(en|it|english|italian)([/_-]|$)', re.IGNORECASE)
drug_dirs = sorted({p.parent for p in find(DATA / 'multicardioner', '*.ann')
                    if 'drug' in str(p.relative_to(DATA)).lower()
                    and not LANG_DIR.search(p.parent.relative_to(DATA).as_posix())
                    and not ner_data._EXCLUDED_DIRS.search(str(p))})

print('DisTEMIST TSV:', *dis_tsvs, sep='\n  ')
print('PharmaCoNER brat:', *pharma_dirs, sep='\n  ')
print('SympTEMIST TSV:', *sym_tsvs, sep='\n  ')
print('DrugTEMIST brat:', *drug_dirs, sep='\n  ')
assert dis_tsvs and sym_tsvs and drug_dirs, 'Revisa las rutas: algún corpus no se encontró'
assert pharma_dirs or not USE_PHARMACONER, 'PharmaCoNER no se encontró'

Revisa la salida anterior antes de seguir:
- **SympTEMIST:** deben aparecer solo los TSV de la subtarea de NER (`SINTOMA`), uno por split. Si se cuela otro TSV, por ejemplo uno de *linking* con códigos, quítalo de `sym_tsvs`.
- **DrugTEMIST:** deben aparecer solo las carpetas brat de medicamentos en español, de entrenamiento y test. Si se cuela una de CardioCCC (cardiología), puedes dejarla: usa las mismas guías.

In [ ]:
# 5. Cargar y fusionar por documento. Se usan todos los splits oficiales con anotación gold (train, dev y test)
#    porque aquí no se reportan métricas sobre ellos: la validación de la etapa 1 es un 10 % aparte.
stats = Counter()

def merge_splits(*parts):
    merged = {}
    for part in parts:
        merged.update(part)
    return merged

symptemist = merge_splits(*[
    ner_data.load_tsv_corpus(tsv, ner_data.pick_text_dir(DATA / 'symptemist', ner_data.read_mentions_tsv(tsv)),
                             'symptemist', stats) for tsv in sym_tsvs])
distemist = merge_splits(*[
    ner_data.load_tsv_corpus(tsv, ner_data.pick_text_dir(tsv.parent.parent, ner_data.read_mentions_tsv(tsv)),
                             'distemist', stats) for tsv in dis_tsvs])
drugtemist = merge_splits(*[ner_data.load_brat_corpus(d, 'drugtemist', stats) for d in drug_dirs])
corpora = {'symptemist': symptemist, 'distemist': distemist, 'drugtemist': drugtemist}
if USE_PHARMACONER:
    corpora['pharmaconer'] = merge_splits(*[ner_data.load_brat_corpus(d, 'pharmaconer', stats) for d in pharma_dirs])

for name, corpus in corpora.items():
    print(f'{name}: {len(corpus)} documentos')

public_docs = ner_data.merge_corpora(list(corpora.values()), required=set(corpora), stats=stats)
print('Fusionados con todas las capas:', ner_data.corpus_summary(public_docs))
print(dict(stats))
assert len(public_docs) > 300, 'Muy pocos documentos comunes: revisa las rutas y los nombres de fichero'

En `stats`, `documento_parcial_descartado` cuenta los documentos a los que les falta alguno de los corpus usados, y `drugtemist:otro_idioma` los ficheros en inglés o italiano que se ignoraron. No se usan porque les faltaría alguna capa de anotación. `texto_distinto` y `span_no_reubicado` indican diferencias de versión entre corpus; si son muchos, conviene revisarlos antes de entrenar.

In [ ]:
# 6. Etapa 1: corpus públicos. Validación con un 10 % de documentos apartados (F1 de seqeval por token).
from transformers import set_seed
set_seed(SEED)

STAGE1 = dict(epochs=3, learning_rate=5e-5, batch_size=16, max_length=512, stride=128, seed=SEED)
tokenizer = train_ner.load_tokenizer()
stage1_train, stage1_val = ner_data.holdout(public_docs, fraction=0.1, seed=SEED)
print('train', len(stage1_train), '| val', len(stage1_val))

model = train_ner.new_model()
trainer1 = train_ner.train(model, tokenizer, stage1_train, stage1_val, str(WORK / 'etapa1'), **STAGE1)
stage1_val_spans = train_ner.evaluate(trainer1.model, tokenizer, stage1_val)
print('Etapa 1, validación (spans exactos):', stage1_val_spans['micro'])
trainer1.save_model(str(WORK / 'etapa1_modelo'))
tokenizer.save_pretrained(str(WORK / 'etapa1_modelo'))

In [ ]:
# 7. Transferencia sin adaptar: modelo de la etapa 1 sobre las transcripciones de dev del corpus propio.
own_dev = ner_data.corpus_to_documents(CODE / 'corpus_entidades.json', split='dev')
print('dev propio:', ner_data.corpus_summary(own_dev))
zero_shot = train_ner.evaluate(trainer1.model, tokenizer, own_dev)
for label, values in zero_shot.items():
    print(f'{label:<11}', values)

In [ ]:
# 8. Etapa 2: validación cruzada de 5 particiones sobre dev (cada partición parte del modelo de la etapa 1).
import json, statistics

STAGE2 = dict(epochs=10, learning_rate=2e-5, batch_size=8, max_length=512, stride=128, seed=SEED)
by_id = {d.doc_id: d for d in own_dev}
folds = ner_data.kfold(list(by_id), k=5, seed=SEED)
fold_reports = []
for index, (train_ids, val_ids) in enumerate(folds, start=1):
    fold_model = train_ner.new_model(str(WORK / 'etapa1_modelo'))
    trainer = train_ner.train(fold_model, tokenizer, [by_id[i] for i in train_ids], None,
                              str(WORK / f'etapa2_fold{index}'), **STAGE2)
    report = train_ner.evaluate(trainer.model, tokenizer, [by_id[i] for i in val_ids])
    fold_reports.append(report)
    print(f'partición {index}: micro-F1 {report["micro"]["f1"]}')
    del trainer, fold_model
    torch.cuda.empty_cache()

def summary(key, label='micro'):
    values = [r[label][key] for r in fold_reports if r[label][key] is not None]
    return {'media': round(statistics.mean(values), 3), 'desviacion': round(statistics.pstdev(values), 3)} if values else None

cv = {label: {k: summary(k, label) for k in ('precision', 'recall', 'f1')}
      for label in (*ner_data.ENTITY_LABELS, 'micro')}
print(json.dumps(cv, indent=2, ensure_ascii=False))

In [ ]:
# 9. Modelo final: etapa 1 + todo dev. Se guarda con sus métricas y su configuración.
import datetime

final_model = train_ner.new_model(str(WORK / 'etapa1_modelo'))
final_trainer = train_ner.train(final_model, tokenizer, own_dev, None, str(WORK / 'etapa2_final'), **STAGE2)

metrics = {
    'etapa1_validacion_spans': stage1_val_spans,
    'transferencia_sin_adaptar_dev': zero_shot,
    'etapa2_validacion_cruzada_dev_5_particiones': cv,
    'nota': 'El split test del corpus propio no se ha usado. Se evalua una vez en el paso 4.1.',
}
config = {
    'modelo_base': train_ner.BASE_MODEL,
    'etiquetas': ner_data.LABELS,
    'etapa1': {**STAGE1, 'documentos_train': len(stage1_train), 'documentos_val': len(stage1_val),
               'corpus_usados': sorted(corpora),
               'corpus': {name: url for name, (url, _) in DATASETS.items()}, 'estadisticas_fusion': dict(stats)},
    'etapa2': {**STAGE2, 'documentos': len(own_dev), 'particiones': 5},
    'semilla': SEED,
    'fecha': datetime.date.today().isoformat(),
    'versiones': {'torch': torch.__version__, 'transformers': transformers.__version__},
}
OUTPUT = WORK / 'ai_triage_ner_roberta'
train_ner.save_artifacts(final_trainer, tokenizer, str(OUTPUT), metrics, config)
print(sorted(p.name for p in OUTPUT.iterdir()))

## 10. Exportar el modelo (nunca a git)

Hay dos opciones:
- **ZIP**, para adjuntarlo a una *release* de GitHub o descargarlo. Ejecuta la siguiente celda.
- **Hugging Face Hub, repositorio privado.** Guarda un token con permiso de escritura en los secretos de Colab con el nombre `HF_TOKEN`, pon `PUSH_TO_HUB = True` y ajusta `HUB_REPO`.

Después, en el backend, `NER_MODEL_PATH` apuntará a la carpeta descomprimida (paso 3.2).

In [ ]:
import shutil
archive = shutil.make_archive(str(WORK / 'ai_triage_ner_roberta'), 'zip', OUTPUT)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    print('En Kaggle: descarga el ZIP desde la pestaña Output.')

PUSH_TO_HUB = False
HUB_REPO = 'TU_USUARIO/ai-triage-ner-roberta'
if PUSH_TO_HUB:
    from huggingface_hub import HfApi
    from google.colab import userdata
    api = HfApi(token=userdata.get('HF_TOKEN'))
    api.create_repo(HUB_REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path=str(OUTPUT), repo_id=HUB_REPO)
    print('Subido a', HUB_REPO, '(privado)')

## Qué reportar en la tesis

- **Etapa 1:** F1 por spans exactos en la validación apartada. Mide la tarea en texto clínico escrito.
- **Transferencia sin adaptar sobre dev:** cuánto se pierde al pasar de texto escrito a habla transcrita.
- **Etapa 2:** media ± desviación de la validación cruzada de 5 particiones en dev.
- **Cifra principal:** la de `test`, que se calcula una sola vez en el paso 4.1 con el sistema híbrido completo.

Hay varias limitaciones que conviene declarar:
- El corpus propio es pequeño y está anotado con asistencia de IA, con revisión humana pendiente.
- La etiqueta ENFERMEDAD agrupa diagnósticos del episodio y antecedentes.
- Los corpus públicos son casos clínicos escritos, no diálogos.